# 03.01 — Bằng chứng Student Main-Core

Câu hỏi: Phase 3 đã freeze những Student nào để Phase 4 dùng? Trả lời dựa trên `main_core_decision.json`, metric validation và review đã khóa. Notebook chỉ đọc evidence; không tạo target, train Student, ghi metric, sửa review hoặc tạo artifact Phase 4.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'evidence/phase03/main_core_decision.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy repository root hoặc Phase 3 decision.')

decision_path = ROOT / 'evidence/phase03/main_core_decision.json'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
assert decision['status'] == 'completed_with_two_frozen_directions_for_phase04'

main_cores = []
for direction, item in decision['decisions'].items():
    main_cores.append({
        'chiều': direction,
        'run': item['run_id'],
        'Student': item['student']['model_id'],
        'checkpoint': item['checkpoint']['path'],
        'checkpoint SHA-256 tree': item['checkpoint']['sha256_tree'],
    })
pd.DataFrame(main_cores).sort_values('chiều')


In [ ]:
rows = []
for direction, item in decision['decisions'].items():
    for dataset, evaluation in item['evaluation'].items():
        student = evaluation['student_metrics']
        teacher = evaluation['teacher_comparable_metrics']
        rows.append({
            'chiều': direction,
            'tập validation': dataset,
            'số câu': evaluation['rows'],
            'Student chrF++': student['chrF++'],
            'Teacher chrF++': teacher['chrF++'],
            'Student SacreBLEU': student['sacreBLEU'],
            'Teacher SacreBLEU': teacher['sacreBLEU'],
        })
validation = pd.DataFrame(rows).sort_values(['chiều', 'tập validation'])
validation


In [ ]:
reviews = []
for direction, item in decision['decisions'].items():
    review = item['manual_review']
    reviews.append({
        'chiều': direction,
        'file review': review['path'],
        'số dòng': review['rows'],
        'lỗi dịch': review['translation_error_yes'],
        'lỗi kỹ thuật': review['technical_error_yes'],
    })
pd.DataFrame(reviews).sort_values('chiều')


Kết quả trên là evidence của Phase 3. Student thấp hơn Teacher trên các cohort có thể so sánh; decision freeze không chứng minh Student ngang Teacher hoặc sẵn sàng chạy Android. Người thực hiện Phase 4 ở máy khác chỉ lấy checkpoint và hash từ decision theo `docs/phase_04/README.md`. Repository này không tạo artifact Phase 4.